# Otimização — capítulo 2: Otimização unidimensional

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aguilarmadeira/optimization-course/blob/main/code/python/02_univariate_optimization/02_5_newton/ex02_5_newton_safeguarded.ipynb)

Corra a célula **Preparação** e depois o exemplo (no Colab: *Runtime → Run all*). No fim, o exemplo compara os resultados com os slides (*confere com os slides: sim*). Os slides usam vírgula decimal; aqui usa-se o ponto. Caderno gerado a partir de `ex02_5_newton_safeguarded.py` por `notebooks/make_notebooks.py`.

*Code comments and printed messages are in Portuguese.*

## Preparação

In [1]:
# Preparação: põe as funções da UC no caminho do Python.
# No Colab (ou noutra pasta), clona o repositório; dentro do repositório, usa code/python.
import os, sys
LOCAL = [p for p in ("..", os.path.join("..", "..")) if os.path.isfile(os.path.join(p, "uc_setup.py"))]
if LOCAL:
    RAIZ = os.path.abspath(LOCAL[0])
else:
    if not os.path.isdir("optimization-course"):
        !git clone -q --depth 1 https://github.com/aguilarmadeira/optimization-course
    RAIZ = os.path.abspath(os.path.join("optimization-course", "code", "python"))
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import uc_setup  # acrescenta as pastas dos métodos e common/
print("Código da UC pronto.")

Código da UC pronto.


## 2.5 Método de Newton–Raphson salvaguardado

Exemplo [`ex02_5_newton_safeguarded.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/02_univariate_optimization/02_5_newton/ex02_5_newton_safeguarded.py) · funções da UC: [`newton_safeguarded.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/02_univariate_optimization/02_5_newton/newton_safeguarded.py)

```text
Reproduz o exemplo «Newton salvaguardado: bisseção + Newton» do deck 2.5.

f'(x) = arctan x (mínimo de f(x) = x arctan x - ln(1 + x^2)/2 em 0),
intervalo [-1, 2], x0 = 1.5 -- o caso em que Newton puro diverge
(página «Quando falha»).  Um passo de bisseção, depois três de Newton:

  k   x_k        passo de Newton               x_k+1        novo intervalo
  0   1.5        -1.694: fora de (a,b) -> bis.  0.5          [-1; 0.5]
  1   0.5        -0.0796: aceite                -0.0796      [-0.0796; 0.5]
  2   -0.0796    3.4e-4: aceite                 3.4e-4       [-0.0796; 3.4e-4]
  3   3.4e-4     -2.5e-11: aceite               -2.5e-11

Paragem |f'| < tolg = 1e-10 (a do script das figuras).
Imprime a tabela, as contagens e a comparação com Newton puro, e no fim
compara com os slides (vírgula decimal nos slides, ponto aqui).
```

In [2]:
import numpy as np

from newton_safeguarded import newton_safeguarded


def confere_sig(v, s, m):
    """v confere com s mostrado com m algarismos significativos?"""
    v = np.ravel(np.asarray(v, float)); s = np.ravel(np.asarray(s, float))
    m = np.ravel(np.asarray(m, float))
    e = np.floor(np.log10(np.abs(s)))
    return bool(np.all(np.abs(v - s) <= 0.5 * 10.0 ** (e - m + 1) * (1 + 1e-6)))


def simnao(c):
    return tuple("sim" if ci else "não" for ci in c)


ok = True
print("Otimização — deck 2.5: Newton salvaguardado (bisseção + Newton)")

f = lambda x: x * np.arctan(x) - 0.5 * np.log(1 + x**2)
df = lambda x: np.arctan(x)
ddf = lambda x: 1 / (1 + x**2)

# ---------------------------------------------- Newton puro diverge
x0 = 1.5
print("\nNewton puro para f'(x) = arctan x a partir de x0 = %g (3 iterações):" % x0)
xs = [x0]
for _ in range(3):                    # x_{k+1} = x_k - f'(x_k)/f''(x_k)
    xs.append(xs[-1] - df(xs[-1]) / ddf(xs[-1]))
print("  x_k: " + " -> ".join("%.4f" % v for v in xs) + "  (|x_k| cresce: diverge)")
c = [abs(xs[3]) > abs(xs[2]) > abs(xs[1]) > abs(xs[0])]
print("  diverge: %s" % simnao(c))
ok = ok and all(c)

# ---------------------------------------------- Newton salvaguardado
a, b, tolg = -1.0, 2.0, 1e-10
print("\nNewton salvaguardado em [%g, %g], x0 = %g, tolg = %g:" % (a, b, x0, tolg))
r = newton_safeguarded(f, df, ddf, a, b, x0, tolg, 50, verbose=True)
H = r.history
print("%s; x* ~ %.1e, f(x*) = %.1e" % (r.message, r.x, r.fx))
print("n_g = %d (f'(a), f'(b), f'(x0) e uma por iteração), n_H = %d, n_f = %d" % (r.ngev, r.nhev, r.nfev))
c = [H.shape[0] == 4,
     list(H[:, 3]) == [1, 0, 0, 0],
     confere_sig(H[:, 1], [1.5, 0.5, -0.0796, 3.4e-4], [2, 1, 3, 2]),
     confere_sig(H[:, 2], [-1.694, -0.0796, 3.4e-4, -2.5e-11], [4, 3, 2, 2]),
     confere_sig(H[:, 4], [0.5, -0.0796, 3.4e-4, -2.5e-11], [1, 3, 2, 2]),
     confere_sig(H[:3, 5:7], [[-1, 0.5], [-0.0796, 0.5], [-0.0796, 3.4e-4]], [[1, 1], [3, 1], [3, 2]]),
     r.nbis == 1, r.ngev == 7, r.nhev == 4]
print("  4 iterações: %s | 1 bisseção + 3 Newton: %s | x_k: %s | passos de Newton: %s | x_k+1: %s"
      " | intervalos: %s | 1 bisseção: %s | n_g = 7: %s | n_H = 4: %s" % simnao(c))
ok = ok and all(c)

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Otimização — deck 2.5: Newton salvaguardado (bisseção + Newton)

Newton puro para f'(x) = arctan x a partir de x0 = 1.5 (3 iterações):
  x_k: 1.5000 -> -1.6941 -> 2.3211 -> -5.1141  (|x_k| cresce: diverge)
  diverge: sim

Newton salvaguardado em [-1, 2], x0 = 1.5, tolg = 1e-10:
  k          x_k     x_Newton      tipo        x_k+1                   [a, b]   df(x_k+1)
  0   1.5000e+00  -1.6941e+00  bisseção   5.0000e-01  [-1.000e+00;  5.000e-01]   +4.64e-01
  1   5.0000e-01  -7.9560e-02    Newton  -7.9560e-02  [-7.956e-02;  5.000e-01]   -7.94e-02
  2  -7.9560e-02   3.3530e-04    Newton   3.3530e-04  [-7.956e-02;  3.353e-04]   +3.35e-04
  3   3.3530e-04  -2.5131e-11    Newton  -2.5131e-11  [-2.513e-11;  3.353e-04]   -2.51e-11
|f'(x)| < tolg em 4 iterações (1 de bisseção); x* ~ -2.5e-11, f(x*) = 6.3e-22
n_g = 7 (f'(a), f'(b), f'(x0) e uma por iteração), n_H = 4, n_f = 0
  4 iterações: sim | 1 bisseção + 3 Newton: sim | x_k: sim | passos de Newton: sim | x_k+1: sim | intervalos: sim | 1 bis